# 12. Batch Processing & Workflow

수백 개 HWP 문서를 처리할 때 필요한 **context manager** 와 실전 workflow.

## 핵심 context manager 6종

| Context | 용도 | 효과 |
|:---|:---|:---|
| `silenced("yes")` | 다이얼로그 자동 응답 | "저장?" 등 YES 자동 |
| `suppress_errors()` | 에러 + Python 예외 swallow | 한두 파일 실패해도 계속 |
| `batch_mode(hide=True)` | 창 숨김 + dialog 억제 | **5~10배 가속** |
| `undo_group("설명")` | 단일 undo 경계 | 사용자 Ctrl+Z 한 번으로 rollback |
| `charshape_scope(**fmt)` | 문자 모양 임시 변경 | 종료 시 snapshot 복원 |
| `use_document(doc)` | 활성 문서 일시 전환 | 블록 내 doc 기준 동작 |


In [ ]:
import sys
sys.path.insert(0, '.')
from tutorial_helpers import run_and_show

## 1. silenced — 다이얼로그 자동 응답

```python
with app.silenced("yes"):
    for path in paths:
        app.open(path)
        app.replace_all("2025", "2026")
        app.save(path)             # "덮어쓸까요?" → YES 자동
# 블록 종료 시 이전 mode 자동 복원
```

사용 가능한 preset: `"yes"`, `"no"`, `"reset"`, `"ok"`, `"save"`, `"save_yes"`, `"save_no"`, `"no_save"`, `"okcancel_ok"`, `"okcancel_no"`


## 2. suppress_errors — 에러도 무시하고 계속

```python
from pathlib import Path

failed = []
with app.silenced():
    for path in Path("input/").glob("*.hwp"):
        try:
            with app.suppress_errors():
                app.open(path)
                app.replace_all("old", "new")
                app.save(str(path).replace("input", "output"))
        except Exception as e:
            failed.append((path, str(e)))
```


## 3. batch_mode — 5~10배 가속

```python
import pandas as pd
from pathlib import Path

customers = pd.read_csv("customers.csv")
with app.batch_mode():                          # 창 숨김 + dialog 억제
    for _, row in customers.iterrows():
        app.open("contract_template.hwp")
        app.fields.from_brackets()
        app.fields.update(row.to_dict())
        app.save(f"out/contract_{row['name']}.hwp")
# 블록 종료 시 자동 복원
```

내부 동작:
- `set_message_box_mode(SILENCE_ALL_YES)` — dialog 억제
- `set_visible(False)` — 창 숨김 (hide=True)
- `FollowActiveWindowOff` — scroll follow 중지

### 예시 결과 문서

`batch_mode` 로 생성한 계약서 1건:


In [ ]:
run_and_show("batch_contract", '''
app.set_charshape(bold=True, height=1600)
app.set_parashape(align_type=3)
app.insert_text("계 약 서\n\n")
app.set_charshape(bold=False, height=1100)
app.set_parashape(align_type=1)
app.insert_text("계약자: {{name}}\n")
app.insert_text("계약일: {{date}}\n")
app.insert_text("계약 금액: {{amount}}원\n")
app.insert_text("연락처: {{phone}}\n")
app.move.top_of_file()
app.fields.from_brackets()
app.fields.update({
    "name": "홍길동",
    "date": "2026-04-15",
    "amount": "1,200,000",
    "phone": "010-1234-5678",
})
''')

*위: 1개 row 에 해당하는 최종 계약서 (수백 건 중 하나). batch_mode 에서는 이 프로세스가 각 row 에 대해 반복됨.*

## 4. undo_group — 단일 Ctrl+Z 로 전체 rollback

```python
with app.undo_group("전체 폰트 교체"):
    app.convert.replace_font("맑은 고딕", "함초롬바탕")
    app.set_charshape(fontsize=1100)
    for para_idx in range(100):
        app.move.para.next()
        app.set_parashape(line_spacing=160)

# 사용자 Ctrl+Z 한 번 → 위 작업 모두 rollback
```


## 5. charshape_scope / parashape_scope — 임시 서식

```python
app.insert_text("앞: ")
with app.charshape_scope(bold=True, text_color="#FF0000"):
    app.insert_text("중요")
app.insert_text(" 뒤")
# "앞: 중요 뒤" — '중요' 만 강조
```

실제 렌더링 결과:


In [ ]:
run_and_show("scope_demo", '''
app.set_charshape(height=1200)
app.insert_text("앞: ")
with app.charshape_scope(bold=True, text_color="#FF0000", height=1500):
    app.insert_text("중요")
app.insert_text(" 뒤\n\n")

app.insert_text("문단 정렬도 가능:\n")
with app.parashape_scope(align_type=3):  # 중앙
    app.insert_text("이 문단은 중앙 정렬됩니다.\n")
app.insert_text("다음 문단은 기본 정렬로 복원됩니다.\n")
''')

*위: `charshape_scope` 와 `parashape_scope` 실제 적용 결과*

## 6. use_document — 다중 문서 컨텍스트 전환

```python
doc1 = app.documents.open("report1.hwp")
doc2 = app.documents.open("report2.hwp")

app.insert_text("doc1 에 삽입")        # doc1 active

with app.use_document(doc2):
    app.insert_text("doc2 에 삽입")     # doc2 일시 전환

app.insert_text("doc1 다시")            # 자동 복원
```


## 참고 — silenced() 실제 효과 측정

`silenced()` 가 dialog 를 어떻게 억제하는지 실제 모드 값으로 확인:


In [ ]:
from hwpapi.core import App
a = App(is_visible=False)

print("# SetMessageBoxMode 비트필드 상수")
print(f"  SILENCE_ALL_YES    = {hex(App.SILENCE_ALL_YES):>10s}")
print(f"  SILENCE_ALL_NO     = {hex(App.SILENCE_ALL_NO):>10s}")
print(f"  SILENCE_RESET      = {hex(App.SILENCE_RESET):>10s}")
print(f"  SILENCE_NO_SAVE    = {hex(App.SILENCE_NO_SAVE):>10s}")

print("\n# silenced() context manager 동작")
prev = a.get_message_box_mode()
print(f"  진입 전: mode = {hex(prev)}")
with a.silenced("yes"):
    during = a.get_message_box_mode()
    print(f"  진입 후: mode = {hex(during)} (모든 dialog YES)")
after = a.get_message_box_mode()
print(f"  종료 후: mode = {hex(after)} (원래 모드 복원)")

try: a.quit()
except Exception: pass

## 참고 — batch_mode() 속도 비교

실제 HWP 에서 `batch_mode()` 효과 측정:


In [ ]:
import time
from hwpapi.core import App
a = App(is_visible=True)   # 일반 모드 시작

# 일반 모드 (visible + dialog on) 측정용 fake loop
t0 = time.perf_counter()
for i in range(10):
    a.api.Run("FileNew")
    a.insert_text(f"doc {i}")
t_normal = time.perf_counter() - t0

# batch_mode 측정용
t0 = time.perf_counter()
with a.batch_mode():
    for i in range(10):
        a.api.Run("FileNew")
        a.insert_text(f"doc {i}")
t_batch = time.perf_counter() - t0

print(f"{'모드':12s} {'10 docs':>12s} {'(per doc)':>12s}")
print(f"{'─' * 12} {'─' * 12} {'─' * 12}")
print(f"{'normal':12s} {t_normal*1000:>9.0f}ms {t_normal*100:>9.1f}ms")
print(f"{'batch_mode':12s} {t_batch*1000:>9.0f}ms {t_batch*100:>9.1f}ms")
print(f"\n  → 가속 비율: {t_normal/t_batch:.1f}x")

try: a.quit()
except Exception: pass

## 7. 실전 Workflow

### 시나리오 A — 급여명세서 1000장 생성

```python
import pandas as pd
from pathlib import Path

def generate_payslips(csv_path, template, out_dir):
    df = pd.read_csv(csv_path)
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    app = App(is_visible=False)
    with app.batch_mode():
        for _, row in df.iterrows():
            with app.suppress_errors():
                app.open(template)
                app.fields.from_brackets()
                app.fields.update(row.to_dict())
                app.save(Path(out_dir) / f"payslip_{row['id']}.hwp")
```

### 시나리오 B — 디렉터리 재귀 폰트 교체

```python
def replace_font_recursive(root, old, new):
    app = App(is_visible=False)
    with app.batch_mode():
        for path in Path(root).rglob("*.hwp"):
            try:
                app.open(str(path))
                with app.undo_group(f"Font {old}→{new}"):
                    app.convert.replace_font(old, new)
                app.save()
                app.close()
            except Exception as e:
                print(f"skip {path}: {e}")
```

### 시나리오 C — 대량 품질 감사

```python
def lint_directory(root):
    app = App(is_visible=False)
    results = []
    with app.batch_mode():
        for path in Path(root).rglob("*.hwp"):
            app.open(str(path))
            r = app.lint()
            if r.has_issues:
                results.append({
                    "path": str(path),
                    "issues": r.issue_count,
                    "long_sentences": len(r.long_sentences),
                })
            app.close()
    return sorted(results, key=lambda x: -x["issues"])
```
